In [ ]:
# --- CLUSTERIZAÇÃO: PREPARAÇÃO ---
# Por que normalizar? O K-Means mede distância entre pontos. Se uma coluna vai de 0 a 100 e outra vai de 0 a 100000, a segunda vai dominar o modelo injustamente. O StandardScaler coloca tudo na mesma escala.

import sys
from pathlib import Path

import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model_utils import (
    preparar_clusterizacao,
    avaliar_k_elbow_silhouette,
    treinar_kmeans,
    rotular_clusters,
    plotar_zonas_vulnerabilidade,
 )

# Features que representam risco social combinado
features_cluster = [
    'vazio_sanitario',           # déficit de infraestrutura
    'Taxa_Morbidade_100k_Hab',   # impacto na saúde
    'indice_tratamento_esgoto',  # qualidade do esgoto
    'indice_perda_distribuicao_agua',  # ineficiência da rede
    'RISCO_SOCIAL_FINAL'         # índice composto
 ]

df_cluster, X_scaled, scaler, ano_ref = preparar_clusterizacao(df_final, features_cluster)

print("✅ Dataset pronto para clusterização.")
print(f"   Ano de referência: {ano_ref}")
print(f"   Municípios com dados completos: {len(df_cluster)}")

In [ ]:
# --- MÉTODO DO COTOVELO ---
# Como funciona: Testamos de 2 a 9 clusters e medimos o "erro interno" (inércia). O ponto onde a curva dobra como um cotovelo é o número ideal — a partir daí, adicionar mais clusters traz pouco ganho.

inercias, silhouettes, _, _ = avaliar_k_elbow_silhouette(X_scaled)
plt.show()

print("\n💡 Escolha o k onde a inércia 'dobra' E a silhueta é alta.")
print("   Geralmente entre 3 e 5 clusters para dados municipais.")

In [ ]:
# --- TREINAR K-MEANS ---
# Ajuste K_FINAL com base no gráfico da célula anterior
K_FINAL = 4

df_cluster, perfil, km_final = treinar_kmeans(
    df_cluster, X_scaled, features_cluster, K_FINAL
 )

# Perfil de cada cluster (médias das features originais)
print("📊 PERFIL DOS CLUSTERS (médias por grupo):\n")
print(perfil.to_string())

In [ ]:
# --- NOMEAR CLUSTERS (ajuste os rótulos conforme o perfil gerado acima) ---
# Ordena clusters por RISCO_SOCIAL_FINAL crescente e atribui rótulos
df_cluster, rotulos = rotular_clusters(df_cluster, perfil, K_FINAL)

# Distribuição
print("\n📊 DISTRIBUIÇÃO DOS MUNICÍPIOS POR ZONA:\n")
print(df_cluster['zona_vulnerabilidade'].value_counts().to_string())

# Scatter: Vazio Sanitário vs Morbidade, colorido por zona
plotar_zonas_vulnerabilidade(df_cluster, ano_ref)
plt.show()

# Propaga os rótulos para o df_final
df_final = df_final.merge(
    df_cluster[['id_municipio', 'cluster', 'zona_vulnerabilidade']],
    on='id_municipio', how='left'
 )
print("\n✅ Zonas propagadas para df_final.")